## Saving latent features to the dataframe

In [4]:
# import libraries
import numpy as np # linear algebra

import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import time
import sys
import glob
import os

import torch
import torch.utils.data
from torch import nn, optim
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from torchvision.utils import save_image
#from torchsummary import summary
#from torchinfo import summary

import matplotlib.pyplot as plt
from PIL import Image
import nibabel as nib

import re
import subprocess

In [1]:
date = '26_05_25'
path = '/csc/epitkane/home/atagmazi/ADDL_pipeline/scripts/bimodal_VAE/' + date

In [2]:
#subprocess.run(['cp', path+'/addl_models_01_12_24.py', '.'], check=True)
#subprocess.run(['cp', path+'/addl_models.py', '.'], check=True)


In [3]:
meta2=  pd.read_csv(path + '/metafile_shuffled_'+ date+'.csv')

NameError: name 'pd' is not defined

In [4]:
# Prep brain mask
brain_mask = nib.load('/csc/epitkane/home/atagmazi/tpl-MNI152NLin6Asym_res-01_desc-brain_T1w.nii.gz').get_fdata()
brain_mask[brain_mask != 0] = 1
#brain_mask = np.stack([brain_mask] * 2, axis=-1)

NameError: name 'nib' is not defined

In [9]:
stat = np.load("/csc/epitkane/home/atagmazi/ADDL_pipeline/scripts/stats_train.npz")
p_quant90 = stat['p_quant90']
m_quant90 = stat['m_quant90']

p_quant95 = stat['p_quant95']
m_quant95 = stat['m_quant95']

p_quant99 = stat['p_quant99']
m_quant99 = stat['m_quant99']

p_quant999 = stat['p_quant999']
m_quant999 = stat['m_quant999']

p_std = stat['p_std']
m_std = stat['m_std']

p_mean_clip = stat['p_mean_clip']
m_mean_clip = stat['m_mean_clip']
p_std_clip = stat['p_std_clip']
m_std_clip = stat['m_std_clip']

p_min_clip = stat['p_mim_clip']
m_min_clip = stat['m_min_clip']
p_max_clip = stat['p_max_clip']
m_max_clip = stat['m_max_clip']

In [10]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model_up=torch.load(path+"/best_model.pth",map_location=torch.device("cuda:0" if torch.cuda.is_available() else "cpu"))


[NbConvertApp] Converting notebook addl_models_bimodel_pytorch.ipynb to script
[NbConvertApp] Writing 26382 bytes to addl_models_bimodel_pytorch.py


In [11]:
model_up.eval()

VAE_1modality(
  (conv): Sequential(
    (0): ConvBlock(
      (conv): Conv2d(1, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (normalize): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (activation): LeakyReLU(negative_slope=0.01)
    )
    (1): ConvBlock(
      (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (normalize): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (activation): LeakyReLU(negative_slope=0.01)
    )
  )
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (dense_mean): Linear(in_features=262144, out_features=64, bias=True)
  (dense_log_var): Linear(in_features=262144, out_features=64, bias=True)
  (dense): Linear(in_features=64, out_features=262144, bias=True)
  (deconv): Sequential(
    (0): DeconvBlock(
      (deconv): ConvTranspose2d(64, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (normalize): BatchNorm2d(32, eps=1e-05, momentum

In [12]:
#split data on train and test subsets
train_size = 0.8
train_end = int(len(meta2)*train_size)

In [13]:
data_train = meta2[:train_end]
data_test = meta2[train_end:]

In [14]:
# create dictionaries with info for generator
data_partition = dict()
#abeta_labels = dict()
#abeta_all = dict()

t = int(0.8*np.shape(data_train)[0]) #!!
v = int(0.2*np.shape(data_train)[0]) 
#DATADIR = r"/csc/epitkane/data/ADNI/AD_DL_03_11_2021/"

# check the path 
data_partition['train'] = meta2.loc[:t,:].reset_index(drop=True) #!!
data_partition['validation'] = meta2.loc[t+1:train_end,:].reset_index(drop=True) #!!
data_partition['test'] = meta2.loc[train_end+1:,:].reset_index(drop=True) #!!

data_partition['all'] = meta2.loc[:,:].reset_index(drop=True) #!!


In [15]:
class PETSliceDataset(Dataset):
    def __init__(self, list_IDs_pet, slice_axis=2, brain_mask=None, 
                 pet_minimum= p_min_clip, pet_maximum=p_max_clip,
                 mri_minimum= m_min_clip, mri_maximum=m_max_clip,
                 pet_quant = p_quant999,mri_quant = m_quant999, 
                 pet_mean = p_mean_clip,mri_mean = m_mean_clip,
                 pet_std = p_std_clip,mri_std = m_std_clip,
                 sagittal_dim=182, coronal_dim=218, axial_dim=182):
        """
        PyTorch Dataset for paired 2D slices of PET and MRI scans.
        """
        self.list_IDs_pet = list_IDs_pet
        #self.list_IDs_mri = list_IDs_mri
        self.slice_axis = slice_axis  # 0 = sagittal, 1 = coronal, 2 = axial
        self.brain_mask = brain_mask
        
        self.pet_minimum = pet_minimum
        self.pet_maximum = pet_maximum
        self.pet_quant = pet_quant
        self.pet_mean = pet_mean
        self.pet_std = pet_std
        self.mri_minimum = mri_minimum
        self.mri_maximum = mri_maximum
        self.mri_quant = mri_quant
        self.mri_mean = mri_mean
        self.mri_std = mri_std
        
        self.sagittal_dim = sagittal_dim
        self.coronal_dim = coronal_dim
        self.axial_dim = axial_dim
        self.slices = self.load_all_slices()  # Preload slice paths
        self.indices = list(range(len(self.slices)))

    def load_all_slices(self):
        """Extracts and pairs 2D slices from all PET/MRI scans."""
        slices = []
        slice_id = 0
        for pet_path in zip(self.list_IDs_pet):
            if self.slice_axis == 0:  # Sagittal
                num_slices = slice_id + self.sagittal_dim 
            elif self.slice_axis == 1:  # Coronal
                num_slices = slice_id + self.coronal_dim 
            else:  # Axial (default)
                num_slices = slice_id + self.axial_dim 

            for within_img_num, i in enumerate(range(slice_id, num_slices)):
                slices.append((pet_path, i, within_img_num))  # Store slice index
            slice_id = num_slices
        return slices

    def __len__(self):
        """Returns the number of slices."""
        return len(self.slices)

    def __data_generation(self, batch_slices):
        """Generates one batch of 2D slices."""
        pet_slices = []
        pet_ids = []
        batch_data = []

        #pet_path, slice_idx, slice_num_inimg = batch_slices[0]
        for slice_info in batch_slices:
            pet_path, slice_idx, slice_num_inimg = slice_info  # Ensure correct unpacking
            pet_path = pet_path[0]
            if not isinstance(pet_path, str):  
                print(pet_path)
                print(slice_idx)
                print(slice_num_inimg)
                raise ValueError(f"Expected pet_path to be a string, got {type(pet_path)}")
                
        
        img_pet = nib.load(pet_path).get_fdata()
        #img_mri = nib.load(mri_path).get_fdata()

            # Extract the corresponding 2D slice
        if self.slice_axis == 0:  # Sagittal
            pet_slice = img_pet[slice_num_inimg, :, :]
            #mri_slice = img_mri[slice_num_inimg, :, :]
            if self.brain_mask is not None:
                bm = self.brain_mask[slice_num_inimg, :, :]
                pet_slice *= bm
                #mri_slice *= bm
                    
        elif self.slice_axis == 1:  # Coronal
            pet_slice = img_pet[:, slice_num_inimg, :]
            #mri_slice = img_mri[:, slice_num_inimg, :]
            if self.brain_mask is not None:
                bm = self.brain_mask[:, slice_num_inimg, :]
                pet_slice *= bm
                #mri_slice *= bm
            
        else:  # Axial (default)
            pet_slice = img_pet[:, :, slice_num_inimg]
            #mri_slice = img_mri[:, :, slice_num_inimg]
            if self.brain_mask is not None:
                bm = self.brain_mask[:, :, slice_num_inimg]
                pet_slice *= bm
                #mri_slice *= bm

            # Skip slices with NaNs or empty regions
        #if np.isnan(pet_slice).any() and np.isnan(mri_slice).any():
        #    return None, None, None, None 
        #if np.max(pet_slice) == 0 and np.max(mri_slice) == 0:
        #    return None, None, None, None 
            # Skip empty slices
            #if pet_slice.size == 0 or mri_slice.size == 0:
            #    continue  # Skip this slice

            # Normalize if necessary (optional step, currently not applied)
        
        pet_norm = self.min_max_normalize(np.asarray(pet_slice, dtype=np.float32), float(self.pet_quant), self.pet_minimum, self.pet_maximum)
        #mri_norm = self.min_max_normalize(np.asarray(mri_slice, dtype=np.float32), float(self.mri_quant), self.mri_minimum, self.mri_maximum)
        
        pet_norm = np.clip(pet_norm, 1e-7, 1 - 1e-7) #???or just to 0 and 1?
            # Stack PET and MRI as channels
        #input_2ch = np.stack([pet_norm, mri_norm], axis=0)
            #print(input_2ch.shape) [2,100,200]

            #batch_data.append(input_2ch)  # Store the correct 2-channel data
        #pet_ids.append(pet_path)
        #mri_ids.append(mri_path)
            #print(f"pet_slice shape: {pet_slice.shape}, mri_slice shape: {mri_slice.shape}")

        # Convert to NumPy arrays
        
        batch_data = np.array(pet_norm, dtype=np.float32)
        #print(batch_data.shape[0])
        batch_data = batch_data.reshape(1, batch_data.shape[0],  batch_data.shape[1])
        #print(batch_data.shape)
        #if len(batch_data) == 0:
            #return None, None, None
        

        # Convert to tensor
        return batch_data, pet_path, slice_num_inimg 

    def __getitem__(self, index):
        """Generates one slice of 2D images (per slice, not batch)."""
        # Get a single slice metadata
        pet_path, slice_idx, slice_num_inimg = self.slices[index]
        
        # Generate single slice data
        X, pet_ids, slice_n = self.__data_generation([(pet_path, slice_idx, slice_num_inimg)])
        X = torch.tensor(X, dtype=torch.float32)
        #print(X.shape)


        return { 'image': X, 'pet_ID': pet_ids, 'slice_number': slice_n }

    #def min_max_normalize(image, min_val, max_val):
    #    return (image - min_val) / (max_val - min_val + 1e-8)  # Avoid division by zero

    def quantile_norm(self, image, quantile, mean, std):
        image = np.asarray(image, dtype=np.float32).copy()  # Ensure array and prevent in-place modification
        #image = np.clip(image, 0, quantile)  # Clip values above quantile
        #image = (image - mean) / (std + 1e-8)  # Avoid division by zero
        return image
    def min_max_normalize(self,image,quantile, min_val, max_val):
        image = np.asarray(image, dtype=np.float32).copy()  # Ensure array and prevent in-place modification
        image = np.clip(image, 0, quantile)  # Clip values above quantile
        return (image - min_val) / (max_val - min_val + 1e-8)  # Avoid division by zero


In [16]:
# creating datasets 
train_dataset = PETSliceDataset(list_IDs_pet = data_partition['train']['PET_PATH_normalised'], brain_mask = brain_mask)

validation_dataset = PETSliceDataset(list_IDs_pet = data_partition['validation']['PET_PATH_normalised'], brain_mask = brain_mask)

test_dataset = PETSliceDataset(list_IDs_pet = data_partition['test']['PET_PATH_normalised'], brain_mask = brain_mask)


In [17]:
# creating dataloaders
train_dataloader = DataLoader(train_dataset, batch_size=64,
                        shuffle=False, num_workers=4)

validation_dataloader = DataLoader(validation_dataset, batch_size=64,
                        shuffle=False, num_workers=4)

test_dataloader = DataLoader(test_dataset, batch_size=64,
                        shuffle=False, num_workers=4)

/csc/epitkane/home/atagmazi/.conda/envs/atagmazi_gpu8/lib/python3.7/site-packages/torch/utils/data/dataloader.py:557: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  cpuset_checked))


In [18]:
#train_dataset[0]['pet_ID']

'I1324526_normalised.nii'

In [71]:
#i

0

In [72]:
#standart loss function
def loss_function(recon_x, x, mu, logvar):
    #recon_loss = F.binary_cross_entropy(recon_x, x,reduction='sum')
    # Mask out the 0 regions
    #mask = (x != 0)  # Create a binary mask for non-zero values

    # Compute loss only for non-zero regions
    recon_loss = F.mse_loss(recon_x, x, reduction='mean') # mean of mse losses within batch 
    #recon_loss = F.binary_cross_entropy(recon_x, x, reduction='mean')
    #recon_loss = torch.mean(torch.mean((x - recon_x) ** 2, dim=(1, 2, 3)))
    #print('Reconstruction loss of the epoch:',recon_loss.mean().item())
    logvar = torch.clamp(logvar, max=5, min = -5)
    KLD = 0 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim = 1) # or torch.mean()?
    #KLD = 0 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    #KLD = 1 * torch.mean(torch.sum(KLD, dim=0))
    #print('KLD loss of the epoch:',KLD.mean().item())
    return recon_loss,torch.mean(KLD)
   


In [74]:
'''df = pd.DataFrame()
for i, (data) in enumerate(train_dataloader):
    while i == 0:
        print(f"Batch {i} Type: {type(data)}")
        print(f"Batch {i} Content: {data}")
    
        ID = data['pet_ID']
        slice_num = data['slice_number']
        data = data['image'].type(torch.FloatTensor).to(device)
        recon_batch, mu, logvar = model_up(data)
        #recon, kld = loss_function(recon_batch, data, mu, logvar)
        #loss = recon + torch.mean(kld)
        #print(loss)
        
        #print(i)
        #train_loss += loss.detach().cpu().numpy()
        for j in range(len(mu)):
            df[str(ID[j]+'_'+str(slice_num[j].detach().cpu().numpy()))] = mu[j].detach().cpu().numpy()
            #df['ID',row] = ID[0][j]
            #row += 1
    break
        #mu_all.append(mu.detach().cpu().numpy())
        #recon_all.append(recon_batch.detach().cpu().numpy())
'''

Batch 0 Type: <class 'dict'>
Batch 0 Content: {'image': tensor([[[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]],


        [[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]],


        [[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          ...,
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.]]],


        ...,


        [[[0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0., 0., 0.],
          [0., 0., 0.,  ..., 0.,

IndexError: too many indices for tensor of dimension 4

In [ ]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
model_up.eval()
#train_loss = []
df = pd.DataFrame()
for i, (data) in enumerate(train_dataloader):
    ID = data['pet_ID']
    slice_num = data['slice_number']
    #cond = data['condition'].type(torch.FloatTensor).to(device)
    data = data['image'].type(torch.FloatTensor).to(device)
    #labels = one_hot(labels, 10)
    recon_batch, mu, logvar = model_up(data)
    recon, kld = loss_function(recon_batch, data, mu, logvar)
    loss = recon + torch.mean(kld)
    #print(loss)
    #print(ID)
    #print(i)
    #train_loss += loss.detach().cpu().numpy()
    data_dict = {}

    for j in range(len(ID)):
        col_name = f"{ID[j]}_{slice_num[j].detach().cpu().numpy()}"
        data_dict[col_name] = mu[j].detach().cpu().numpy()  # Convert tensor to NumPy
    
    # Convert dictionary to DataFrame and join efficiently
    new_df = pd.DataFrame(data_dict)
    
    # Merge with existing DataFrame
    df = pd.concat([df, new_df], axis=1)
    #for j in range(len(mu)):
        #df[str(ID[j]+'_'+str(slice_num[j].detach().cpu().numpy()))] = mu[j].detach().cpu().numpy()
        #df['ID',row] = ID[0][j]
        #row += 1
    #mu_all.append(mu.detach().cpu().numpy())
    #recon_all.append(recon_batch.detach().cpu().numpy())
    
df.to_csv(path + '/latent_features_train2_'+ date+'.csv')
print('Train latent features were saved!')

In [ ]:
#device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
model_up.eval()
df = pd.DataFrame()
for i, (data) in enumerate(validation_dataloader):
    ID = data['pet_ID']
    slice_num = data['slice_number']
    #cond = data['condition'].type(torch.FloatTensor).to(device)
    data = data['image'].type(torch.FloatTensor).to(device)
    #labels = one_hot(labels, 10)
    recon_batch, mu, logvar = model_up(data)
    recon, kld = loss_function(recon_batch, data, mu, logvar)
    loss = recon + torch.mean(kld)
    #print(loss)
    #print(ID)
    #print(i)
    #test_loss += loss.detach().cpu().numpy()
    data_dict = {}

    for j in range(len(ID)):
        col_name = f"{ID[j]}_{slice_num[j].detach().cpu().numpy()}"
        data_dict[col_name] = mu[j].detach().cpu().numpy()  # Convert tensor to NumPy
    
    # Convert dictionary to DataFrame and join efficiently
    new_df = pd.DataFrame(data_dict)
    
    # Merge with existing DataFrame
    df = pd.concat([df, new_df], axis=1)
    #for j in range(len(mu)):
        #df[str(ID[j]+'_'+str(slice_num[j].detach().cpu().numpy()))] = mu[j].detach().cpu().numpy()
        #df['ID',row] = ID[0][j]
        #row += 1
    #mu_all.append(mu.detach().cpu().numpy())
    #recon_all.append(recon_batch.detach().cpu().numpy())
    
df.to_csv(path + '/latent_features_validation2_'+ date+'.csv')
print('Validation latent features were saved!')

In [ ]:
#device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)
model_up.eval()
df = pd.DataFrame()
for i, (data) in enumerate(test_dataloader):
    ID = data['pet_ID']
    slice_num = data['slice_number']
    #cond = data['condition'].type(torch.FloatTensor).to(device)
    data = data['image'].type(torch.FloatTensor).to(device)
    #labels = one_hot(labels, 10)
    recon_batch, mu, logvar = model_up(data)
    recon, kld = loss_function(recon_batch, data, mu, logvar)
    loss = recon + torch.mean(kld)
    #print(loss)
    #print(ID)
    #print(i)
    #test_loss += loss.detach().cpu().numpy()
    data_dict = {}

    for j in range(len(ID)):
        col_name = f"{ID[j]}_{slice_num[j].detach().cpu().numpy()}"
        data_dict[col_name] = mu[j].detach().cpu().numpy()  # Convert tensor to NumPy
    
    # Convert dictionary to DataFrame and join efficiently
    new_df = pd.DataFrame(data_dict)
    
    # Merge with existing DataFrame
    df = pd.concat([df, new_df], axis=1)
    #for j in range(len(mu)):
        #df[str(ID[j]+'_'+str(slice_num[j].detach().cpu().numpy()))] = mu[j].detach().cpu().numpy()
        #df['ID',row] = ID[0][j]
        #row += 1
    #mu_all.append(mu.detach().cpu().numpy())
    #recon_all.append(recon_batch.detach().cpu().numpy())
    
df.to_csv(path + '/latent_features_test2_'+ date+'.csv')
print('Test latent features were saved!')